In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import os

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

2026-04-28 19:13:38.237782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777403618.616925      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777403618.720666      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777403619.690348      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403619.690399      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777403619.690403      55 computation_placer.cc:177] computation placer alr

In [ ]:
CHECKPOINT_FILEPATH = '/kaggle/working/chkpts/checkpoint.model.keras'
TUNED_CHECKPOINT_FILEPATH = '/kaggle/working/chkpts/tuned-checkpoint.model.keras'

In [ ]:
# dataset augmentation
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.15),
])

model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# freeze weights
model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)
x = model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(23, activation='softmax')(x)

model = keras.Model(inputs, outputs)

I0000 00:00:1777403663.845031      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777403663.850967      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │        29,463 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,079,034 (15.56 MB)

 Trainable params: 29,463 (115.09 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
labeled = pd.read_csv("/kaggle/input/datasets/jkong05/c-logo-labels/labels.csv")
labeled["sector"] = labeled["sector"].astype("category")
labeled["label_idx"] = labeled["sector"].cat.codes

classes = labeled["sector"].cat.categories
num_classes = len(classes)

train_df, temp_df = train_test_split(labeled, test_size=0.2, stratify=labeled["sector"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["sector"], random_state=42)

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    
    # do not need to /= 255 because EfficientNetB0 already has a layer that rescales the input
    return tf.cast(img, tf.float32), label

train_data = tf.data.Dataset.from_tensor_slices((train_df["filepath"].values, train_df["label_idx"].values))
train_data = train_data.shuffle(1000).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

val_data = tf.data.Dataset.from_tensor_slices((val_df["filepath"].values, val_df["label_idx"].values))
val_data = val_data.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

test_data = tf.data.Dataset.from_tensor_slices((test_df["filepath"].values, test_df["label_idx"].values))
test_data = test_data.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', 
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_SparseTopKCategoricalAccuracy'),
    ]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_FILEPATH,
        monitor="val_loss",
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode="min",
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [ ]:
'''
loading base model from checkpoint for fine-tuning
    - uncomment the line below and re-run following cells for base model training
'''
model = keras.models.load_model("/kaggle/input/models/jkong05/b0-base-best-frozen-23-classifications/keras/default/1")

In [ ]:
# H = model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)

Epoch 1/20
5126/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640

2026-04-28 19:24:33.099991: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.237965: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.553611: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:33.695667: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:24:34.509188: E external/local_xla/xla/stream_

5127/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640

2026-04-28 19:25:51.217259: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.359820: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.709253: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:51.851758: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-28 19:25:52.634989: E external/local_xla/xla/stream_


Epoch 1: val_loss improved from inf to 2.98957, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 511s 96ms/step - accuracy: 0.1108 - loss: 3.0243 - top_3_accuracy: 0.2640 - val_accuracy: 0.1196 - val_loss: 2.9896 - val_top_3_accuracy: 0.2792
Epoch 2/20
5125/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1179 - loss: 3.0069 - top_3_accuracy: 0.2762
Epoch 2: val_loss improved from 2.98957 to 2.98634, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 191s 37ms/step - accuracy: 0.1179 - loss: 3.0069 - top_3_accuracy: 0.2762 - val_accuracy: 0.1269 - val_loss: 2.9863 - val_top_3_accuracy: 0.2858
Epoch 3/20
5125/5127 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1195 - loss: 3.0032 - top_3_accuracy: 0.2787
Epoch 3: val_loss improved from 2.98634 to 2.98589, saving model to /kaggle/working/chkpts/checkpoint.model.keras
5127/5127 ━━━━━━━━━━━━━━━━━━━━ 190s 37ms/step - accuracy: 0.1195 - loss: 3.0032 -

In [ ]:
def create_accuracy_loss_plot(H):
    acc = H.history['accuracy']
    val_acc = H.history['val_accuracy']
    loss = H.history['loss']
    val_loss = H.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, acc, label='Train')
    ax1.plot(epochs, val_acc, label='Val')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(epochs, loss, label='Train')
    ax2.plot(epochs, val_loss, label='Val')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.tight_layout()
    plt.show()

def create_confusion_matrix(model, test_data, classes):
    preds = model.predict(test_data).argmax(axis=1)
    true_labels = np.concatenate([y for _, y in test_data], axis=0)

    cm = confusion_matrix(true_labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)

    fig, ax = plt.subplots(figsize=(16, 16))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    plt.tight_layout()
    plt.show()
    

def create_heatmap(img_array, model):
    base = model.get_layer('efficientnetb0')

    inner_grad_model = tf.keras.Model(
        inputs=base.inputs,
        outputs=[base.get_layer('top_activation').output, base.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, base_out = inner_grad_model(img_array, training=False)
        tape.watch(conv_outputs)

        x = model.get_layer('global_average_pooling2d')(base_out)
        x = model.get_layer('dropout')(x, training=False)
        predictions = model.get_layer('dense')(x)

        pred_idx = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_idx]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_idx.numpy()


def display_heatmap(img_path, true_idx, model, classes):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img_array = tf.expand_dims(tf.cast(img, tf.float32), 0)

    heatmap, pred_idx = create_heatmap(img_array, model)

    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    orig = np.uint8(img.numpy())
    overlay = cv2.addWeighted(orig, 0.6, heatmap, 0.4, 0)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
    ax1.imshow(orig)
    ax1.set_title('Original') 
    ax1.axis('off')
    ax2.imshow(heatmap) 
    ax2.set_title('Heatmap') 
    ax2.axis('off')
    ax3.imshow(overlay)
    ax3.set_title(f'Pred: {classes[pred_idx]}\nTrue: {classes[true_idx]}')
    ax3.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# create_accuracy_loss_plot(H)

In [ ]:
# create_confusion_matrix(model, test_data, classes)

In [ ]:
# for img_path, true_idx in zip(test_df['filepath'].iloc[:5], test_df['label_idx'].iloc[:5]):
#     display_heatmap(img_path, true_idx, model, classes)

In [ ]:
# tuning
base = model.get_layer('efficientnetb0')
base.trainable = True

for layer in base.layers[:-20]:
    layer.trainable = False

for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_SparseTopKCategoricalAccuracy'),
    ]
)

tuned_callbacks = [
    keras.callbacks.ModelCheckpoint(
        TUNED_CHECKPOINT_FILEPATH,
        monitor="val_loss",
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode="min",
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [ ]:
H2 = model.fit(train_data, validation_data=val_data, epochs=10, callbacks=tuned_callbacks)

In [ ]:
create_accuracy_loss_plot(H2)

In [ ]:
create_confusion_matrix(model, test_data, classes)

In [ ]:
for img_path, true_idx in zip(test_df['filepath'].iloc[:5], test_df['label_idx'].iloc[:5]):
    display_heatmap(img_path, true_idx, model, classes)